In [1]:
import collections
from datetime import datetime, timedelta, timezone
import os
import pprint
import random
import time

import ee
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import openet.core

from IPython.display import Image, display
import ipyplot


In [2]:
ee.Initialize(
    project='openet-dri',
    opt_url='https://earthengine-highvolume.googleapis.com'
)

# stats_ws = os.path.join(os.getcwd(), 'stats')
# if not os.path.isdir(stats_ws):
#     os.makedirs(stats_ws)
    

In [3]:
cloudmask_stats_coll_id = 'projects/openet/assets/cloudmask/landsat/v2_1'

wrs2_skip_list = [
    'p050r026',                         # Vancouver Island
    'p048r028',                         # OR/WA Coast
    'p042r037',                         # San Nicholas Island, California
    'p040r040', 'p039r040',             # Isla Guadalupe
    'p038r043', 'p036r043',             # Baja coast
    'p019r040', 'p018r040',             # West Florida coast
    'p016r043', 'p015r043',             # South Florida coast
    'p014r041', 'p014r042', 'p014r043', # East Florida coast
    'p013r034', 'p013r035', 'p013r036', # North Carolina Outer Banks
    'p011r032',                         # Rhode Island coast
    'p010r030',                         # Maine
    # Caribbean tiles
    'p013r041', 'p013r042', 'p013r043',             # Bahamas
    'p012r042', 'p012r043', 'p011r042', 'p011r043', # Bahamas
    'p006r037', 'p006r038', # Bermuda
    'p017r044', 'p016r044', 'p015r044', 'p014r044', # Cuba
    'p013r044', 'p012r044', 'p011r044', 'p010r044', # Cuba/Bahamas
]

wrs2_path_skip_list = [
    1, 2, 3, 4, 5, 6,
    7, 8, 9, 
    51, 52, 53, 54, 55
]
wrs2_row_skip_list = [
    20, 21, 22, 23, 24, 
    44, 45
]

wrs2_list = sorted(
    ee.FeatureCollection('projects/openet/assets/features/wrs2/custom')
    .filterBounds(ee.Geometry.BBox(-127, 24.5, -65, 51))
    .filter(ee.Filter.inList('wrs2_tile', wrs2_skip_list).Not())
    .filter(ee.Filter.inList('wrs2_path', wrs2_path_skip_list).Not())
    .filter(ee.Filter.inList('wrs2_row', wrs2_row_skip_list).Not())
    .aggregate_histogram('wrs2_tile').keys().getInfo(),
    reverse=True
)
print(len(wrs2_list))

ocean_wrs2_list = [
    'p048r027', 'p047r031', 'p047r030', 'p047r029', 'p046r033', 
    'p045r034', 'p044r035', 'p043r036', 'p041r037', 'p040r038', 
    'p038r041', 'p038r040',
    'p025r040', 'p024r040', 'p024r027', 'p023r040', 
    'p023r027', 'p022r040', 'p021r040', 'p020r029',
    'p017r041', 'p016r038', 'p015r040', 'p015r037', 
    'p013r033', 'p012r032', 'p011r031', 'p011r030', 
]

california_wrs2_list = [
    'p038r036', 'p038r037', 
    'p039r035', 'p039r036', 'p039r037',
    'p040r034', 'p040r035', 'p040r036', 'p040r037',
    'p041r034', 'p041r035', 'p041r036', 'p041r037',
    'p042r033', 'p042r034', 'p042r035', 'p042r036',
    'p043r031', 'p043r032', 'p043r033', 'p043r034', 'p043r035',
    'p044r031', 'p044r032', 'p044r033', 'p044r034',
    'p045r031', 'p045r032', 'p045r033',
    'p046r031', 'p046r032', 'p047r031',
]


568


In [4]:
land_mask = ee.Image('projects/openet/assets/features/water_mask').Not()
# Apply the NLCD/NALCMS water mask (anywhere it is water, set the ocean mask 
land_mask = land_mask.where(ee.Image("USGS/NLCD_RELEASES/2020_REL/NALCMS").unmask(18).eq(18), 0)
# land_mask = land_mask.And(ee.Image("USGS/NLCD_RELEASES/2020_REL/NALCMS").unmask(18).neq(18))
# # land_mask = ee.Image('projects/openet/assets/meteorology/conus404/ancillary/land_mask')

# etf_coll_id = 'projects/openet/assets/ssebop/conus/gridmet/landsat/c02'
etf_coll_id = 'projects/usgs-gee-nhm-ssebop/assets/ssebop/landsat/c02'
# etf_coll_id = 'projects/openet/assets/intercomparison/ssebop/landsat/c02/v0p2p6'
band_name = 'et_fraction'

rgb_bands = {
    'LT04': ['SR_B3', 'SR_B2', 'SR_B1'],
    'LT05': ['SR_B3', 'SR_B2', 'SR_B1'],
    'LE07': ['SR_B3', 'SR_B2', 'SR_B1'],
    'LC08': ['SR_B4', 'SR_B3', 'SR_B2'],
    'LC09': ['SR_B4', 'SR_B3', 'SR_B2'],
}

# 0 - white, 1 - no fill (green), 2 - shadow (dark blue), 3 - snow (light blue), 4 - cloud (light gray), 5 - water (purple), 6 - ocean mask
fmask_palette = "ffffff, 9effa1, blue, 00aff2, dddddd, purple, bfbfbf"
fmask_max = 6

# # 0 - white, 1 - no fill (green), 2 - shadow (dark blue), 3 - snow (light blue), 4 - cloud (light gray), 5 - water (purple), 6 - saturated mask, 7 - ocean mask
# fmask_palette = "ffffff, 9effa1, blue, 00aff2, dddddd, purple, red, bfbfbf"
# fmask_max = 7


In [5]:
# # Intercomparison sites and dates

# sites_csv = '/Users/Charles.Morton@dri.edu/Projects/openet-tools/intercomparison/master_flux_station_list.csv'
# sites_df = pd.read_csv(sites_csv)

# interp_days = 32
# site_keep_list = []
# wrs2_delimiter = ';'

# # Hardcoding the sites CSV field names for now
# start_field = 'START_DATE'
# end_field = 'END_DATE'
# site_field = 'SITE_ID'
# lat_field = 'LATITUDE'
# lon_field = 'LONGITUDE'
# wrs2_field = 'WRS2_TILES'

# from dateutil.relativedelta import relativedelta

# # Group the date ranges by WRS2 tile
# # print(f'\nGrouping overlapping dates')
# wrs2_dates = collections.defaultdict(list)
# wrs2_sites = collections.defaultdict(list)
# for (site_i, site) in sites_df.iterrows():
#     # print(site_i)
#     # print(site)
#     if site['RANDOM_SELECTION'] not in [0, 1]:
#         # print('  Unsupported RANDOM_SELECTION value')
#         input('ENTER')
#         continue
#     # if site['RANDOM_SELECTION'] != 1:
#     #     continue
#     if site_keep_list and site.loc[site_field] not in site_keep_list:
#         # print('  Site not in keep list - skipping')
#         continue

#     # Include all sites in INI file, even those outside the date range
#     for wrs2 in site.loc[wrs2_field].split(wrs2_delimiter):
#         wrs2_sites[wrs2.strip()].append([
#             round(site.loc[lon_field], 6), round(site.loc[lat_field], 6)
#         ])

#     start_dt = datetime.datetime.strptime(site.loc[start_field], '%Y-%m-%d')
#     end_dt = datetime.datetime.strptime(site.loc[end_field], '%Y-%m-%d')
#     # print(f'  Start Date: {start_dt.strftime("%Y-%m-%d")}')
#     # print(f'  End Date:   {end_dt.strftime("%Y-%m-%d")}')

#     # If start/end dates are within N gap days of the start/end of the month
#     #   consider it a "full" month
#     gap_days = 5
#     # print('  Snapping start date to month')
#     month_start_dt = datetime.datetime(start_dt.year, start_dt.month, 1)
#     if (start_dt - month_start_dt).days <= gap_days:
#         # print('    full month')
#         start_dt = month_start_dt
#     else:
#         # print('    not full month')
#         start_dt = month_start_dt + relativedelta(months=1)

#     # print('  Snapping end date to month')
#     month_end_dt = end_dt + relativedelta(months=1)
#     month_end_dt = datetime.datetime(month_end_dt.year, month_end_dt.month, 1)
#     month_end_dt = month_end_dt - relativedelta(days=1)
#     if (month_end_dt - end_dt).days <= gap_days:
#         # print('    full month')
#         end_dt = month_end_dt
#     else:
#         # print('    not full month')
#         end_dt = (month_end_dt + relativedelta(days=1) -
#                   relativedelta(months=1) - relativedelta(days=1))
#     # print(f'  Start Date: {start_dt.strftime("%Y-%m-%d")}')
#     # print(f'  End Date:   {end_dt.strftime("%Y-%m-%d")}')

#     if interp_days > 0:
#         # Buffer the date ranges by the interpolate days value if set
#         # print('  Buffering start/end dates')
#         start_dt = start_dt - datetime.timedelta(days=interp_days)
#         end_dt = end_dt + datetime.timedelta(days=interp_days)
#         # print(f'  Start Date: {start_dt.strftime("%Y-%m-%d")}')
#         # print(f'  End Date:   {end_dt.strftime("%Y-%m-%d")}')

#     # CM - Changing conditionals to get single date ranges to work
#     # if end_dt <= start_dt or start_dt >= end_dt:
#     if end_dt < start_dt or start_dt > end_dt:
#         # print(f'  Start: {start_dt.strftime("%Y-%m-%d")}')
#         # print(f'  End:   {end_dt.strftime("%Y-%m-%d")}')
#         # print('  Date range outside min/max, skipping')
#         continue
#     else:
#         # print(f'  Start: {start_dt.strftime("%Y-%m-%d")}')
#         # print(f'  End:   {end_dt.strftime("%Y-%m-%d")}')
#         pass

#     for wrs2 in site.loc[wrs2_field].split(wrs2_delimiter):
#         wrs2_dates[wrs2.strip()].append([start_dt, end_dt])

# # pprint.pprint(wrs2_dates)

# # Merge the date ranges that overlap
# print(f'\nMerging overlapping dates')
# merged_dates = {}
# for wrs2, dates in sorted(wrs2_dates.items()):
#     # print(f'  {wrs2}')
#     # pprint.pprint(sorted(dates))

#     # Push the first interval on to the stack
#     merged_dates[wrs2] = [sorted(dates)[0]]

#     # Only check for overlapping ranges if there is more than 1 range
#     if len(dates) == 1:
#         continue

#     for d in sorted(dates)[1:]:
#         # If the current date range doesn't overlap, add it to the stack
#         if d[0] > merged_dates[wrs2][-1][1]:
#             merged_dates[wrs2].append(d)
#         # If the ranges overlap and the end date is later,
#         #   update the end time of the stack value
#         elif ((d[0] <= merged_dates[wrs2][-1][1]) and
#               (d[1] > merged_dates[wrs2][-1][1])):
#             merged_dates[wrs2][-1][1] = d[1]

# # pprint.pprint(merged_dates)

# # # CGM - Splitting by year for DisALEXI is not needed if the NLCD
# # #   is set to the image collection instead of the image
# # # For DisALEXI split the date ranges by year after merging
# # # For other models, index by the first year in the range
# # # This may be functionality we will want for other models later
# # year_dates = collections.defaultdict(dict)
# # # if model in ['DISALEXI_TAIR_10K', 'DISALEXI_TAIR_1K', 'DISALEXI', 'DISALEXI_TAIR_DIRECT']:
# # #     for wrs2, dates in merged_dates.items():
# # #         # split_dates = {}
# # #         for date_i, date in enumerate(dates):
# # #             for year in range(date[0].year, date[1].year+1):
# # #                 year_date = [
# # #                     max(date[0], datetime.datetime(year, 1, 1)),
# # #                     min(date[1], datetime.datetime(year, 12, 31)),
# # #                 ]
# # #                 try:
# # #                     year_dates[wrs2][year].append(year_date)
# # #                 except:
# # #                     year_dates[wrs2][year] = [year_date]
# # # else:
# # for wrs2, dates in merged_dates.items():
# #     year_dates[wrs2][dates[0][0].year] = dates

# # pprint.pprint(year_dates)


In [6]:
def fmask(landsat_img):
    # Add the fmask image on top of the true color image
    qa_img = landsat_img.select('QA_PIXEL')
    fill_mask = qa_img.bitwiseAnd(1).neq(0)                  # bits: 0
    dilate_mask = qa_img.rightShift(1).bitwiseAnd(1).neq(0)  # bits: 1
    cirrus_mask = qa_img.rightShift(2).bitwiseAnd(1).neq(0)  # bits: 2
    cloud_mask = qa_img.rightShift(3).bitwiseAnd(1).neq(0)   # bits: 3
    shadow_mask = qa_img.rightShift(4).bitwiseAnd(1).neq(0)  # bits: 4
    snow_mask = qa_img.rightShift(5).bitwiseAnd(1).neq(0)    # bits: 5
    clear_mask = qa_img.rightShift(6).bitwiseAnd(1).neq(0)   # bits: 6
    water_mask = qa_img.rightShift(7).bitwiseAnd(1).neq(0)   # bits: 7
    cloud_conf = qa_img.rightShift(8).bitwiseAnd(3)          # bits: 8, 9
    shadow_conf = qa_img.rightShift(10).bitwiseAnd(3)        # bits: 10, 11
    snow_conf = qa_img.rightShift(12).bitwiseAnd(3)          # bits: 12, 13
    cirrus_conf = qa_img.rightShift(14).bitwiseAnd(3)        # bits: 14, 15

    # Saturated pixels
    # Flag as saturated if any of the RGB bands are saturated
    #   or change .gt(0) to .gt(7) to flag if all RGB bands are saturated
    # Comment out rightShift line to flag if saturated in any band
    bitshift = ee.Dictionary({'LANDSAT_4': 0, 'LANDSAT_5': 0, 'LANDSAT_7': 0, 'LANDSAT_8': 1, 'LANDSAT_9': 1});
    saturated_mask = (
        landsat_img.select('QA_RADSAT')
        .rightShift(ee.Number(bitshift.get(ee.String(landsat_img.get('SPACECRAFT_ID'))))).bitwiseAnd(7)
        .gt(0)
    )
    
    # Old "Fmask" style image
    fmask_img = (
        qa_img.multiply(0)
        .where(landsat_img.select(['SR_B4']).mask().eq(0), 1)
        # .where(saturated_mask, 6)
        .where(water_mask, 5)
        .where(shadow_mask, 2)
        .where(snow_mask, 3)
        .where(cloud_mask.Or(dilate_mask).Or(cirrus_mask), 4)
        # .add(shadow_mask.multiply(2))
        # .add(snow_mask.multiply(3))
        # .add(cloud_mask.Or(dilate_mask).Or(cirrus_mask).multiply(4))
        # .add(cloud_mask.Or(dilate_mask).multiply(4))
        # .add(cloud_mask.And(cloud_conf).multiply(4))
        # .add(water_mask.multiply(5))
    )
    
    return fmask_img.updateMask(fmask_img.neq(0)).rename(['fmask'])


In [7]:
# Clean up the scene skip list file
skip_path = '../v2p1.csv'
# skip_path = '../v2p1_eemetric_error.csv'
print(f'\n{skip_path}')

with open(skip_path, 'r') as csv_f:
    scene_skip_lines = csv_f.readlines()
scene_skip_header = scene_skip_lines.pop(0)

# Drop the comments and empty lines
scene_skip_lines = [line.strip() for line in scene_skip_lines if line.strip() and line[0] != '#']

# Sort by date then by tile
scene_skip_lines = sorted(scene_skip_lines, key=lambda x:x.split(',')[0].split('_')[-1] + '_' + x.split(',')[0].split('_')[-2])

# Identify duplicate scene IDs (as opposed to duplicate lines)
# Note, this block is not removing any lines, just printing
print('Duplicate Scene IDs:')

if len({l.split(',')[0] for l in scene_skip_lines}) != len(scene_skip_lines):
    for item, count in collections.Counter([l.split(',')[0] for l in scene_skip_lines]).items():
        if count > 1:
            print(item)

# Identify lines with no reason
print('\nMissing reason Scene IDs:')
for l in scene_skip_lines:
    if ',' not in l:
        print(l)
    elif l.split(',')[1].strip() == '':
        print(l)
    elif len(l.split(',')) > 2:
        print(l)

# # Identify duplicate lines (not duplicate SCENE IDs)
# if len({line for line in scene_skip_lines}) != len(scene_skip_lines):
#     print('Duplicate Lines:')
#     for item, count in collections.Counter(scene_skip_lines).items():
#         if count > 1:
#             print(item)
# 
#     # # Uncomment to have the tool remove duplicate lines
#     # scene_remove_lines = []
#     # for item, count in collections.Counter(scene_skip_lines).items():
#     #     if count > 1:
#     #         scene_remove_lines.append(item)
#     #         # print(item)
#      
#     # # Does this only remove the first one?
#     # if scene_remove_lines:
#     #     print(f'Removing {len(scene_remove_lines)} duplicate lines in file')
#     #     for line in scene_remove_lines:
#     #         print(line)
#     #         scene_skip_lines.remove(line)
# 
# # Then recheck for duplicate SCENE_IDs (but different notes or dates)
# scenes = {line.split(',')[0] for line in scene_skip_lines}           
# if len(scenes) != len(scene_skip_lines):
#     print('Duplicate scene IDs still in file')
    
print('\nWriting updated scene skip list CSV')
with open(skip_path.replace('.csv', '_sorted.csv'), 'w') as csv_f:
    csv_f.write(scene_skip_header)
    for i, line in enumerate(scene_skip_lines):
        csv_f.write(line + '\n')

print('\nDone')


../v2p1.csv
Duplicate Scene IDs:

Missing reason Scene IDs:

Writing updated scene skip list CSV

Done


In [8]:
# Build the EEMETRIC skip list by merging the EEMETRIC error list and the full skip list
scene_skip_path = '../v2p1_sorted.csv'
eemetric_error_path = '../v2p1_eemetric_error.csv'
eemetric_skip_path = '../v2p1_eemetric.csv'

with open(eemetric_error_path, 'r') as csv_f:
    eemetric_skip_lines = csv_f.readlines()
    
with open(scene_skip_path, 'r') as csv_f:
    scene_skip_lines = csv_f.readlines()

print('\nWriting eemetric scene skip list CSV')
with open(eemetric_skip_path, 'w') as csv_f:
    for i, line in enumerate(eemetric_skip_lines):
        csv_f.write(line)
    csv_f.write('\n')
    for i, line in enumerate(scene_skip_lines):
        if i == 0:
            continue
        csv_f.write(line)

print('\nDone')


Writing eemetric scene skip list CSV

Done


In [9]:
### Print recent scenes with high masked count percentages
count_threshold_pct_min = 70
count_threshold_pct_max = 101

# count_threshold_pct_min = 70
# count_threshold_pct_max = 91

months = 8
end_date = datetime.now(timezone.utc).strftime('%Y-%m-%d')
start_date = ee.Date(end_date).advance(-months, 'months').format('yyyy-MM-dd').getInfo()

#start_date = '2025-01-01'
#end_date = '2026-04-01'
print(start_date)
print(end_date)

# image_size = 600
image_size = 1000
print_count = 20

# Read in the scene skip list
scene_skip_url = '../v2p1.csv'
# scene_skip_url = 'https://raw.githubusercontent.com/cgmorton/scene-skip-list/main/v2p1.csv'
scene_skip_df = pd.read_csv(scene_skip_url)
scene_skip_list = list(scene_skip_df['SCENE_ID'].values)
print(f'Skip list images: {len(scene_skip_list)}')

# Filter the scene skip list to the same recent months
recent_scene_skip_list = [
    scene_id for scene_id in scene_skip_list
    if datetime.strptime(scene_id.split('_')[-1], '%Y%m%d').strftime('%Y-%m-%d') >= start_date 
]
print(f'Skip list images: {len(recent_scene_skip_list)}')

scene_cloudscore_url = '../v2p1_cloudscore.csv'
# scene_cloudscore_url = 'https://raw.githubusercontent.com/cgmorton/scene-skip-list/main/v2p1_cloudscore.csv'
scene_cloudscore_list = list(pd.read_csv(scene_cloudscore_url)['SCENE_ID'].values)
print(f'Skip cloudscore images: {len(scene_cloudscore_list)}')


def compute_cloud_ratios(img):
    scene_id = ee.String(img.get('SCENE_ID'))
    
    masked_pixels = (
        ee.Number(img.get('CLOUD_PIXELS'))
        .add(ee.Number(img.get('CIRRUS_PIXELS')))
        .add(ee.Number(img.get('DILATE_PIXELS')))
        .add(ee.Number(img.get('SHADOW_PIXELS')))
        .add(ee.Number(img.get('SNOW_PIXELS')))
        .add(ee.Number(img.get('WATER_PIXELS')))
        .add(ee.Number(img.get('ACCA_PIXELS')))
        # .add(ee.Number(img.get('SATURATED_PIXELS')))
    )

    properties = {
        'CLOUD_COUNT_RATIO': masked_pixels.divide(img.get('TOTAL_PIXELS')),
        'ACCA_COUNT_RATIO': ee.Number(img.get('ACCA_PIXELS')).divide(img.get('TOTAL_PIXELS')),
        'SNOW_COUNT_RATIO': ee.Number(img.get('SNOW_PIXELS')).divide(img.get('TOTAL_PIXELS')),
        'SHADOW_COUNT_RATIO': ee.Number(img.get('SHADOW_PIXELS')).divide(img.get('TOTAL_PIXELS')),
        'WATER_COUNT_RATIO': ee.Number(img.get('WATER_PIXELS')).divide(img.get('TOTAL_PIXELS')),
        'ACCA_PIXELS': ee.Number(img.get('ACCA_PIXELS')),
        'CLOUD_PIXELS': ee.Number(img.get('CLOUD_PIXELS')),
        'CIRRUS_PIXELS': ee.Number(img.get('CIRRUS_PIXELS')),
        'DILATE_PIXELS': ee.Number(img.get('DILATE_PIXELS')),
        'SHADOW_PIXELS': ee.Number(img.get('SHADOW_PIXELS')),
        'SNOW_PIXELS': ee.Number(img.get('SNOW_PIXELS')),
        'WATER_PIXELS': ee.Number(img.get('WATER_PIXELS')),
        'TOTAL_PIXELS': ee.Number(img.get('TOTAL_PIXELS')),
        'MASKED_PIXELS': masked_pixels,
        'UNMASKED_PIXELS': ee.Number(img.get('UNMASKED_PIXELS')),
        'CLOUD_COVER_LAND': ee.Number(img.get('CLOUD_COVER_LAND')),
        'DATE': scene_id.slice(12, 20),
        'SCENE_ID': scene_id,
        'WRS2': ee.String('p').cat(scene_id.slice(5, 8)).cat('r').cat(scene_id.slice(8, 11)),
    }
    
    return ee.Feature(None, properties)


# # Get the CLOUD_COVER_LAND values from the collection before mapping
# cloudcover_pct_info = (
#     ee.ImageCollection(cloudmask_stats_coll_id)
#     .filterDate(start_date, end_date)
#     .filter(ee.Filter.inList('wrs2_tile', wrs2_list))
#     .getInfo()


print('\nReading cloud stats from assets')
cloudmask_stats_full_coll = (
    ee.ImageCollection(cloudmask_stats_coll_id)
    .filterDate(start_date, end_date)
    .filter(ee.Filter.inList('wrs2_tile', wrs2_list))
    # .filter(ee.Filter.inList('wrs2_path', wrs2_path_skip_list).Not())
    # .filter(ee.Filter.inList('wrs2_row', wrs2_row_skip_list).Not())
    # .filter(ee.Filter.inList('wrs2_tile', wrs2_skip_list).Not())
    .map(compute_cloud_ratios, False)
)

cloudmask_stats_coll = (
    cloudmask_stats_full_coll
    .filterMetadata('CLOUD_COVER_LAND', 'less_than', 70.1)
    .filter(ee.Filter.inList('SCENE_ID', recent_scene_skip_list).Not())
    .filter(ee.Filter.inList('SCENE_ID', scene_cloudscore_list).Not())
)

# Filter scenes based on the cloud count ratio 
cloudy_scene_info = (
    cloudmask_stats_coll
    .filterMetadata('CLOUD_COUNT_RATIO', 'not_less_than', count_threshold_pct_min/100)
    .filterMetadata('CLOUD_COUNT_RATIO', 'less_than', count_threshold_pct_max/100)
    .sort('CLOUD_COUNT_RATIO', False)
    .limit(print_count)
    .getInfo()
)

# # Filter scenes based on the number of unmasked ACCA pixels
# count_threshold_pct_min = 0
# count_threshold_pct_max = 101
# cloudy_scene_info = (
#     cloudmask_stats_coll
#     .filterMetadata('ACCA_PIXELS', 'greater_than', 10000)
#     .filterMetadata('ACCA_PIXELS', 'less_than', 10000000)
#     .sort('ACCA_PIXELS', False)
#     .limit(print_count)
#     .getInfo()
# )

# # Filter scenes based on the number of unmasked pixels
# count_threshold_pct_min = 0
# count_threshold_pct_max = 101
# cloudy_scene_info = (
#     cloudmask_stats_coll
#     .filterMetadata('UNMASKED_PIXELS', 'less_than', 500000)
#     # .sort('CLOUD_COUNT_RATIO', False)
#     # .sort('UNMASKED_PIXELS', True)
#     .limit(print_count)
#     .getInfo()
# )

# # Filter scenes based on the number of snow masked pixels
# count_threshold_pct_min = 0
# count_threshold_pct_max = 101
# cloudy_scene_info = (
#     cloudmask_stats_coll
#     .filterMetadata('SNOW_PIXELS', 'greater_than', 5000000)
#     # .filterMetadata('SNOW_PIXELS', 'greater_than', 10000000)
#     # .filterMetadata('SNOW_PIXELS', 'greater_than', 20000000)
#     # .sort('CLOUD_COUNT_RATIO', False)
#     .limit(print_count)
#     .getInfo()
# )


# Convert getInfo output to a dictionary
cloudy_scene_info = {ftr['properties']['SCENE_ID']: ftr['properties'] for ftr in cloudy_scene_info['features']}


# Get the CLOUD_COVER_LAND for all the scenes above and below the target scenes
above_id_list = [
    scene_id.upper().replace(f'{scene_id[5:8]}{scene_id[8:11]}', f'{scene_id[5:8]}{int(scene_id[8:11])-1:03d}') 
    for scene_id in cloudy_scene_info.keys()
]
below_id_list = [
    scene_id.upper().replace(f'{scene_id[5:8]}{scene_id[8:11]}', f'{scene_id[5:8]}{int(scene_id[8:11])+1:03d}') 
    for scene_id in cloudy_scene_info.keys()
]
above_scene_info = cloudmask_stats_full_coll.filter(ee.Filter.inList('SCENE_ID', above_id_list)).getInfo()
below_scene_info = cloudmask_stats_full_coll.filter(ee.Filter.inList('SCENE_ID', below_id_list)).getInfo()
above_cloud_cover = {
    ftr['properties']['SCENE_ID']: ftr['properties']['CLOUD_COVER_LAND'] 
    for ftr in above_scene_info['features']
}
below_cloud_cover = {
    ftr['properties']['SCENE_ID']: ftr['properties']['CLOUD_COVER_LAND'] 
    for ftr in below_scene_info['features']
}


# Iterate through scenes
new_skip_scenes = []
# for scene_id, row in sorted(cloudy_scene_info.items(), key=lambda x: random.random()):
for scene_id, row in cloudy_scene_info.items():

    wrs2 = scene_id.split('_')[1]
    wrs2_path = int(wrs2[0:3])
    wrs2_row = int(wrs2[3:6])
    wrs2_tgt = f'{wrs2_path:03d}{wrs2_row:03d}'
    wrs2_above = f'{wrs2_path:03d}{(wrs2_row-1):03d}'
    wrs2_below = f'{wrs2_path:03d}{(wrs2_row+1):03d}'   

    above_scene_id = scene_id.upper().replace(wrs2_tgt, wrs2_above)
    below_scene_id = scene_id.upper().replace(wrs2_tgt, wrs2_below)
    try:
        above_cloud_pct = float(above_cloud_cover[above_scene_id])
    except:
        above_cloud_pct = None
    try:
        below_cloud_pct = float(below_cloud_cover[below_scene_id])
    except:
        below_cloud_pct = None

    # # Only show scenes that have above & below both skipped or None
    # if (((above_scene_id not in recent_scene_skip_list) and (above_cloud_pct is not None) and (above_cloud_pct < 71)) or 
    #     ((below_scene_id not in recent_scene_skip_list) and (below_cloud_pct is not None) and (below_cloud_pct < 71))):
    #     continue   
    
    # # Only show scenes that have either above & below skipped or None
    # if (((above_scene_id not in recent_scene_skip_list) and (above_cloud_pct is not None) and (above_cloud_pct < 71)) and 
    #     ((below_scene_id not in recent_scene_skip_list) and (below_cloud_pct is not None) and (below_cloud_pct < 71))):
    #     continue   
    
    landsat_type = scene_id.split('_')[0].upper()
    landsat_img = ee.Image(f'LANDSAT/{landsat_type}/C02/T1_L2/{scene_id}')
    landsat_region = landsat_img.geometry().bounds(1, 'EPSG:4326')
    landsat_sr_img = landsat_img.select(rgb_bands[landsat_type]).multiply([0.0000275]).add([-0.2])

    # Landsat true color image
    landsat_url = (
        landsat_sr_img.where(land_mask.unmask().eq(0), 0.25)
        .getThumbURL({'min': 0.0, 'max': 0.30, 'gamma': 1.25, 'region': landsat_region, 'dimensions': image_size})
    )
    time.sleep(1)

    # Landsat true color with Fmask
    fmask_url = (
        landsat_sr_img.where(land_mask.unmask().eq(0), 0.25).visualize(min=0, max=0.3, gamma=1.25)
        .blend(fmask(landsat_img).where(land_mask.unmask().eq(0), fmask_max).visualize(bands='fmask', min=0, max=fmask_max, palette=fmask_palette))
        .getThumbURL({'region': landsat_region, 'dimensions': image_size})
    )
    time.sleep(1)

    print('#'*80)
    print(
        f'  {scene_id}  {row["TOTAL_PIXELS"]:>10d}  {row["UNMASKED_PIXELS"]:>10d}'
        f'  ({row["CLOUD_COUNT_RATIO"]:>0.2f}) ({row["SNOW_COUNT_RATIO"]:>0.2f}) {row["CLOUD_COVER_LAND"]}'
        # f'  {row["SR_RED"]:0.2f}  {row["SR_GREEN"]:0.2f}  {row["SR_BLUE"]:0.2f}'
    )
    ipyplot.plot_images([landsat_url, fmask_url], img_width=image_size)

    # Show the images "above" the target wrs2
    above_img_id = f'LANDSAT/{landsat_type}/C02/T1_L2/{above_scene_id}'
    above_img = ee.Image(above_img_id)
    above_region = above_img.geometry().bounds(1, 'EPSG:4326')
    above_sr_img = above_img.select(rgb_bands[landsat_type]).multiply([0.0000275]).add([-0.2])
    try:
        above_url = (
            above_sr_img.where(land_mask.unmask().eq(0), 0.25).visualize(min=0, max=0.3, gamma=1.25)
            .blend(fmask(above_img).where(land_mask.unmask().eq(0), fmask_max).visualize(bands='fmask', min=0, max=fmask_max, palette=fmask_palette))
            .getThumbURL({'region': above_region, 'dimensions': image_size})
        )
    except:
        above_url = None
    time.sleep(1)

    # Show the images "below" the target wrs2
    below_img_id = f'LANDSAT/{landsat_type}/C02/T1_L2/{below_scene_id}'
    below_img = ee.Image(below_img_id)
    below_region = below_img.geometry().bounds(1, 'EPSG:4326')
    below_sr_img = below_img.select(rgb_bands[landsat_type]).multiply([0.0000275]).add([-0.2])
    try:
        below_url = (
            below_sr_img.where(land_mask.unmask().eq(0), 0.25).visualize(min=0, max=0.3, gamma=1.25)
            .blend(fmask(below_img).where(land_mask.unmask().eq(0), fmask_max).visualize(bands='fmask', min=0, max=fmask_max, palette=fmask_palette))
            .getThumbURL({'region': below_region, 'dimensions': image_size})
        )
    except:
        below_url = None
    time.sleep(1)

    above_skipped = f' (skipped)' if above_scene_id in recent_scene_skip_list or ((above_cloud_pct is not None) and (above_cloud_pct >= 71)) else ''   
    below_skipped = f' (skipped)' if below_scene_id in recent_scene_skip_list or ((below_cloud_pct is not None) and (below_cloud_pct >= 71)) else ''
        
    if above_url and below_url:
        # print(f'{below_scene_id}{below_skipped}  {above_scene_id}{above_skipped}')
        print(f'{below_scene_id} ({below_cloud_pct}){below_skipped}  {above_scene_id} ({above_cloud_pct}){above_skipped}')
        ipyplot.plot_images([below_url, above_url], img_width=image_size)
    elif above_url:
        # print(f'{above_scene_id}{above_skipped}')
        print(f'{above_scene_id} ({above_cloud_pct}){above_skipped}')
        ipyplot.plot_images([above_url], img_width=image_size)
    elif below_url:
        # print(f'{below_scene_id}{below_skipped}')
        print(f'{below_scene_id} ({below_cloud_pct}){below_skipped}')
        ipyplot.plot_images([below_url], img_width=image_size)
    
    new_skip_scenes.append(scene_id)

print('\nNew Skip Scenes')
if new_skip_scenes:
    for scene_id in new_skip_scenes:
        print(scene_id)

print('\nDone')

2025-12-31
2026-08-31
Skip list images: 70653
Skip list images: 1483
Skip cloudscore images: 308

Reading cloud stats from assets
################################################################################
  LC09_030041_20260205    39308983     3445609  (0.87) (0.00) 44.44


LC09_030042_20260205 (18.13)  LC09_030040_20260205 (14.97)


################################################################################
  LC08_036041_20260106    22797965     1819394  (0.87) (0.00) 66.31


LC08_036042_20260106 (53.57)  LC08_036040_20260106 (77.23) (skipped)


################################################################################
  LC09_035025_20260208    38878756     3568611  (0.85) (0.76) 5.7


...

################################################################################
  LC09_026032_20260617    40332964     2673356  (0.81) (0.00) 69.6


LC09_026033_20260617 (11.22)  LC09_026031_20260617 (66.99)


################################################################################
  LC08_031038_20260612    40255861     1170087  (0.81) (0.00) 69.62


LC08_031039_20260612 (79.22) (skipped)  LC08_031037_20260612 (50.31)



New Skip Scenes
LC09_030041_20260205
LC08_036041_20260106
LC09_035025_20260208
LC08_040037_20260219
LC09_032030_20260323
LC09_026033_20260313
LC09_031041_20260127
LC08_031043_20260119
LC09_022038_20260723
LC09_033031_20260314
LC08_019032_20260608
LC09_014033_20260221
LC09_019032_20260413
LC09_039033_20260425
LC09_035042_20260413
LC09_012031_20260802
LC09_037032_20260310
LC09_018027_20260422
LC09_026032_20260617
LC08_031038_20260612

Done
